In [13]:
import re
import torch
from dataclasses import dataclass, field

dtype_map = {
    torch.float32: ('float',   lambda x: f"{repr(x)}f"),
    torch.float64: ('double',  lambda x: repr(x)),
    torch.int64:   ('int64_t', lambda x: str(int(x))),
    torch.int32:   ('int32_t', lambda x: str(int(x))),
    torch.int16:   ('int16_t', lambda x: str(int(x))),
    torch.int8:    ('int8_t',  lambda x: str(int(x))),
    torch.uint8:   ('uint8_t', lambda x: str(int(x))),
}

@dataclass(init=True)
class Checkpoint:
    path: str
    keys: list[str] = field(init=False, default_factory=list)
    values: list[torch.Tensor] = field(init=False, default_factory=list)

    def __post_init__(self):
        loaded = torch.load(self.path)
        
        for key, value in loaded['model'].items():
            self.keys.append(key)
            self.values.append(value)

    @staticmethod
    def _make_good_name(name):
        name = re.sub(f"[^0-9a-zA-Z_]", '_', name)

        if name and name[0].isdigit():
            name = "_" + name
        return name.upper()

    @classmethod
    def _nested_format(cls, obj, fmt, indent=0):
        sp = "\t" * indent

        if isinstance(obj, list):
            if not obj:
                return "{}"
            if not isinstance(obj[0], list):
                return "{" + ", ".join(fmt(x) for x in obj) + "}"
            inner = tuple(cls._nested_format(x, fmt, indent+1) for x in obj)
            return "{\n" + ",\n".join("\t" * (indent+1) + s for s in inner) + "\n" + sp + "}"
        else:
            return fmt(obj)

    def export(self, output_path):
        lines = [
            "#pragma once",
            "#include <cstdint>",
            "#include <cstddef>",
            "",
        ]
        for name, tsr in zip(self.keys, self.values):
            name = self._make_good_name(name)
            tsr = tsr.detach().cpu().contiguous()
            
            shape = tuple(tsr.shape)
            dims = "".join(f"[{d}]" for d in shape)
            ndim = len(shape)
            ctype, fmt = dtype_map[tsr.dtype]
            nested = tsr.tolist()
            
            lines.extend([
                f"// {name} shape={shape}, dtype={tsr.dtype}",
                *[f"#define {name}_DIM{i} {d}" for i, d in enumerate(shape)],
                f""
                f"static const {ctype} {name}{dims} = {self._nested_format(nested, fmt, 0)};",
                f""
            ])
        with open(output_path, 'w', encoding='utf-8') as h:
            h.write('\n'.join(lines))

In [ ]:
checkpoint = Checkpoint('./milestones/MNISTSLP-2026-04-24-20-26-42.pth')
checkpoint.export('./weights.h')